# CVP mid-term: train vanilla 3DGS (gsplat) on RealX3D and score its geometry

Works on **Google Colab** (Runtime → Change runtime type → **T4 GPU**) and **Kaggle** (Settings → Accelerator **GPU T4**, and **Internet ON**).

Run the cells top to bottom. Expected times on a free T4:
- install + compile gsplat: ~5–10 min (first time only per session)
- data download (one scene, 2 conditions + laser scan): ~2–4 min
- training 30k steps: roughly 45–90 min per condition (7k steps: ~10–20 min)
- geometry evaluation: ~3–5 min

At the end, download `cvp_results.zip` (all metrics, previews, ply, figures).

In [1]:
# 1) GPU check
!nvidia-smi --query-gpu=name,compute_cap,memory.total --format=csv

name, compute_cap, memory.total [MiB]
Tesla T4, 7.5, 15360 MiB


In [6]:
# 2) Get the code. EITHER clone from GitHub (after you pushed it) ...
import os
REPO_URL = "https://github.com/Saksham-Bali/cvp_project"   # <- your repo
WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else "/content"
os.chdir(WORK)
if not os.path.exists("cvp_project"):
    !git clone -q $REPO_URL
# If the repo is PRIVATE, use https://<your-github-token>@github.com/Saksham-Bali/cvp_project
# ... OR upload cvp_code.zip with the file browser and run:  !unzip -q cvp_code.zip -d cvp_project
os.chdir(f"{WORK}/cvp_project")
!ls

cvp   notebooks  README.md	       requirements-train.txt  scripts
docs  outputs	 requirements-cpu.txt  RUN_MIDTERM.md	       tests


In [7]:
# 3) Install (gsplat compiles its CUDA code on first use; this cell triggers that compile now)
!pip -q install gsplat==1.5.3 ninja jaxtyping rich plyfile huggingface_hub requests lpips
!python scripts/gsplat_smoke.py
# If the compile fails on a new PyTorch: !pip install git+https://github.com/nerfstudio-project/gsplat.git  (then re-run this cell)

Streaming output truncated to the last 5000 lines.
(   ●  ) gsplat: Setting up CUDA with MAX_JOBS=10 (This may take a few minutes 
(  ●   ) gsplat: Setting up CUDA with MAX_JOBS=10 (This may take a few minutes 
( ●    ) gsplat: Setting up CUDA with MAX_JOBS=10 (This may take a few minutes 
( ●    ) gsplat: Setting up CUDA with MAX_JOBS=10 (This may take a few minutes 
(  ●   ) gsplat: Setting up CUDA with MAX_JOBS=10 (This may take a few minutes 
(   ●  ) gsplat: Setting up CUDA with MAX_JOBS=10 (This may take a few minutes 
(    ● ) gsplat: Setting up CUDA with MAX_JOBS=10 (This may take a few minutes 
(     ●) gsplat: Setting up CUDA with MAX_JOBS=10 (This may take a few minutes 
(    ● ) gsplat: Setting up CUDA with MAX_JOBS=10 (This may take a few minutes 
(   ●  ) gsplat: Setting up CUDA with MAX_JOBS=10 (This may take a few minutes 
(  ●   ) gsplat: Setting up CUDA with MAX_JOBS=10 (This may take a few minutes 
( ●    ) gsplat: Setting up CUDA with MAX_JOBS=10 (This may take a fe

In [8]:
# 4) Download ONE scene: quarter-res images for the 2 strong conditions + laser ground truth
#    (+ RealX3D's published 3DGS numbers for the same scene, for the PSNR comparison)
SCENE = "Ujikintoki"
!python scripts/download_realx3d.py --what quarter gt --scenes $SCENE --conditions motion_strong defocus_strong
!python scripts/download_realx3d.py --what baselines --scenes $SCENE --conditions motion_strong

Saving into: /content/cvp_project/data
Scenes: ['Ujikintoki']
Conditions: ['motion_strong', 'defocus_strong']
Parts: ['gt', 'quarter']

[quarter] motion_strong: streaming (keeps only ['Ujikintoki']) ...
    finished Ujikintoki
[quarter] motion_strong: done
[quarter] defocus_strong: streaming (keeps only ['Ujikintoki']) ...
    finished Ujikintoki
[quarter] defocus_strong: done
[gt] Ujikintoki
  downloading pointclouds/Ujikintoki.tar.gz ...

pointclouds/Ujikintoki.tar.gz: downloading bytes:  92% 235M/256M [00:02<00:00, 202MB/s, 19.7MB/s  ]
pointclouds/Ujikintoki.tar.gz: downloading bytes:  80% 256M/321M [00:03<00:00, 138MB/s, 22.4MB/s  ]
pointclouds/Ujikintoki.tar.gz: downloading bytes: 100% 256M/256M [00:03<00:00, 74.3MB/s, 22.5MB/s  ]
pointclouds/Ujikintoki.tar.gz: reconstructing file: 100% 256M/256M [00:03<00:00, 74.2MB/s, 23.2MB/s  ]

All done.
Saving into: /content/cvp_project/data
Scenes: ['Ujikintoki']
Conditions: ['motion_strong']
Parts: ['baselines']

[baselines] motion_strong:

In [9]:
# 5) Train vanilla 3DGS on STRONG MOTION blur.  STEPS=30000 for the real run, 7000 for a quick one.
STEPS = 7000
!python scripts/train_gsplat.py --data data --scene $SCENE --condition motion_strong --steps $STEPS

Scene Ujikintoki / motion_strong / images=blurred: 30 train views, 5 test views, 1758x1168, 35,314 initial Gaussians, scene scale 3.98 m
step      0/7000  loss 0.1952  #gaussians 35,314  elapsed 0.1 min  eta 355.2 min
step    500/7000  loss 0.0264  #gaussians 35,314  elapsed 1.7 min  eta 21.8 min
step   1000/7000  loss 0.0245  #gaussians 58,224  elapsed 3.5 min  eta 20.8 min
step   1500/7000  loss 0.0205  #gaussians 82,943  elapsed 5.3 min  eta 19.4 min
step   2000/7000  loss 0.0152  #gaussians 104,307  elapsed 7.2 min  eta 17.9 min
step   2500/7000  loss 0.0118  #gaussians 122,718  elapsed 9.0 min  eta 16.3 min
step   3000/7000  loss 0.0182  #gaussians 138,773  elapsed 10.9 min  eta 14.6 min
step   3500/7000  loss 0.0226  #gaussians 132,769  elapsed 12.8 min  eta 12.8 min
step   4000/7000  loss 0.0216  #gaussians 144,080  elapsed 14.7 min  eta 11.0 min
step   4500/7000  loss 0.0201  #gaussians 153,632  elapsed 16.5 min  eta 9.2 min
step   5000/7000  loss 0.0216  #gaussians 161,614  el

In [10]:
# 6) Score its geometry against the laser scan, overall and per structure type
!python scripts/eval_run.py --run runs/$SCENE/motion_strong/vanilla/s0 --data data

  laser cloud: 3,966,751 points (0s)
  eval points after 1 cm voxel: 755,159
  observed in >=2 training views: 638,190 (84.5%)  (21s)
  normals + curvature done (40s)
  proximity done (107s)
  saved outputs/gt_cache/Ujikintoki_strata.npz
scoring again with the published-depth protocol (for the comparison figure) ...
Ujikintoki / motion_strong / vanilla:  F@5cm overall 0.618 (P 0.606, R 0.630); acc median 4.0 cm
   curvature  low: 0.543  mid: 0.611  high: 0.690
   proximity  tight: 0.791  near: 0.778  open: 0.576
  published-protocol F@5cm overall 0.748 (use this one when comparing with RealX3D's published methods)
comparison figure: outputs/Ujikintoki/compare_motion_strong_vanilla.png
done (231s)


In [11]:
# 7) (if time) the same for STRONG DEFOCUS
!python scripts/train_gsplat.py --data data --scene $SCENE --condition defocus_strong --steps $STEPS
!python scripts/eval_run.py --run runs/$SCENE/defocus_strong/vanilla/s0 --data data

Scene Ujikintoki / defocus_strong / images=blurred: 30 train views, 5 test views, 1758x1168, 35,314 initial Gaussians, scene scale 3.98 m
step      0/7000  loss 0.1928  #gaussians 35,314  elapsed 0.0 min  eta 119.5 min
step    500/7000  loss 0.0159  #gaussians 35,314  elapsed 1.7 min  eta 22.2 min
step   1000/7000  loss 0.0115  #gaussians 43,308  elapsed 3.5 min  eta 21.0 min
step   1500/7000  loss 0.0091  #gaussians 48,595  elapsed 5.3 min  eta 19.5 min
step   2000/7000  loss 0.0070  #gaussians 52,737  elapsed 7.1 min  eta 17.8 min
step   2500/7000  loss 0.0067  #gaussians 55,376  elapsed 9.0 min  eta 16.1 min
step   3000/7000  loss 0.0068  #gaussians 57,221  elapsed 10.8 min  eta 14.4 min
step   3500/7000  loss 0.0073  #gaussians 55,953  elapsed 12.6 min  eta 12.6 min
step   4000/7000  loss 0.0065  #gaussians 56,934  elapsed 14.4 min  eta 10.8 min
step   4500/7000  loss 0.0067  #gaussians 57,705  elapsed 16.2 min  eta 9.0 min
step   5000/7000  loss 0.0059  #gaussians 58,359  elapsed 

In [ ]:
# 8) (optional, if time) SHARP CONTROL: same views but the sharp reference images
!python scripts/train_gsplat.py --data data --scene $SCENE --condition motion_strong --images sharp --steps $STEPS
!python scripts/eval_run.py --run runs/$SCENE/motion_strong/sharp_control/s0 --data data

Scene Ujikintoki / motion_strong / images=sharp: 30 train views, 5 test views, 1758x1168, 35,314 initial Gaussians, scene scale 3.98 m
step      0/7000  loss 0.2340  #gaussians 35,314  elapsed 0.0 min  eta 122.3 min
step    500/7000  loss 0.0491  #gaussians 35,314  elapsed 1.8 min  eta 23.1 min
step   1000/7000  loss 0.0377  #gaussians 83,367  elapsed 3.6 min  eta 21.4 min
step   1500/7000  loss 0.0314  #gaussians 141,245  elapsed 5.4 min  eta 19.9 min
step   2000/7000  loss 0.0276  #gaussians 191,843  elapsed 7.3 min  eta 18.3 min
step   2500/7000  loss 0.0266  #gaussians 234,256  elapsed 9.3 min  eta 16.7 min
step   3000/7000  loss 0.0283  #gaussians 269,332  elapsed 11.2 min  eta 15.0 min
step   3500/7000  loss 0.0240  #gaussians 256,601  elapsed 13.2 min  eta 13.2 min
step   4000/7000  loss 0.0246  #gaussians 280,463  elapsed 15.2 min  eta 11.4 min
step   4500/7000  loss 0.0214  #gaussians 299,990  elapsed 17.2 min  eta 9.5 min


In [ ]:
# 9) Look at a preview: blurred input | our render | sharp reference | rendered depth
import glob
from IPython.display import Image, display
for p in sorted(glob.glob(f"runs/{SCENE}/*/*/s0/previews/*.jpg"))[:4]:
    print(p); display(Image(p, width=1100))

In [ ]:
# 10) Zip everything small enough to download (metrics, previews, geometry, ply)
!zip -q -r cvp_results.zip runs outputs -x "*/depth/*"
!ls -lh cvp_results.zip
# Colab: download from the file browser (left), or:
try:
    from google.colab import files; files.download("cvp_results.zip")
except Exception:
    print("On Kaggle: find cvp_results.zip under Output (/kaggle/working)")